In [31]:
import pandas as pd

df = pd.read_csv("ALL_CR_LLM_new.csv")
# feature_cols = df.columns[1:25].tolist() + [df.columns[37]] # Financial
# feature_cols = df.columns[1:38].tolist() # Financial + GPT
feature_cols = df.columns[1:38].tolist() + df.columns[47:].tolist() # Financial + GPT + LLM

In [32]:
features = df[feature_cols]
features

,P/E mean,P/E standard deviation,P/B mean,P/B standard deviation,P/S mean,P/S standard deviation,P/CF mean,P/CF standard deviation,EV/EBITDA mean,EV/EBITDA standard deviation,...,max_avg,min_avg,mean_avg,median_avg,std_avg,max_var,min_var,mean_var,median_var,std_var
0,19.564186,14.126322,3.435766,3.681150,3.702945,2.991120,9.117738,8.070590,8.407772,8.217485,...,0.034858,0.000000,0.022628,0.021868,0.009176,0.000416,0.000000,0.000185,0.000178,0.000098
1,19.564186,14.126322,3.435766,3.681150,3.702945,2.991120,9.117738,8.070590,8.407772,8.217485,...,0.057404,0.024570,0.035154,0.032179,0.009785,0.002206,0.000161,0.000474,0.000263,0.000588
2,19.564186,14.126322,3.435766,3.681150,3.702945,2.991120,9.117738,8.070590,8.407772,8.217485,...,0.052590,0.026000,0.036955,0.033183,0.009685,0.001462,0.000193,0.000404,0.000257,0.000364
3,19.564186,14.126322,3.435766,3.681150,3.702945,2.991120,9.117738,8.070590,8.407772,8.217485,...,0.053577,0.018160,0.031766,0.029163,0.011465,0.001821,0.000129,0.000505,0.000279,0.000504
4,19.564186,14.126322,3.435766,3.681150,3.702945,2.991120,9.117738,8.070590,8.407772,8.217485,...,0.055700,0.017963,0.032845,0.028731,0.011207,0.001508,0.000157,0.000561,0.000464,0.000383
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
757,27.153672,37.636080,-3.203255,43.779844,4.001678,3.395551,15.986967,16.341411,-29.899013,243.156438,...,0.080700,0.014720,0.056943,0.057407,0.016229,0.002617,0.000070,0.000505,0.000352,0.000478
758,27.153673,37.636083,-3.203256,43.779847,4.001678,3.395551,15.986967,16.341412,-29.899012,243.156438,...,0.078933,0.014229,0.058468,0.064243,0.017459,0.001928,0.000062,0.000476,0.000383,0.000377
759,27.402956,129.401877,10.532332,35.620072,3.182172,2.577177,15.542689,8.301743,-3.628943,87.190640,...,0.040883,0.012867,0.026994,0.025957,0.007291,0.000341,0.000040,0.000158,0.000151,0.000078
760,27.402956,129.401876,10.532330,35.620061,3.182172,2.577177,15.542689,8.301743,-3.628944,87.190640,...,0.080567,0.020137,0.054392,0.055667,0.015603,0.001742,0.000096,0.000405,0.000308,0.000319


In [33]:
results = df.iloc[:, 38:47]
results

,EW,MCW,MVO,MS-M,OH-M,MS-T,OH-T,MS-L,OH-L
0,1.182096,1.180342,1.396442,1.178499,1.299723,1.183579,1.268241,1.182502,1.277979
1,1.946927,2.115514,2.431064,1.943876,2.552811,1.952399,2.433478,1.951101,2.665844
2,1.615736,1.583079,2.179303,1.599746,2.263794,1.600876,1.826763,1.630723,2.399198
3,2.713212,3.080356,2.010811,2.861708,5.874684,2.714018,3.609375,2.830019,4.857925
4,4.417240,5.558497,4.482721,5.739160,16.116393,5.347088,8.961685,6.264498,14.877607
...,...,...,...,...,...,...,...,...,...
757,1.048516,1.048590,1.018984,1.171159,1.193730,1.006649,0.993979,1.025081,1.146835
758,1.223088,1.243881,1.341740,1.158452,1.323826,1.090205,1.044297,0.925409,0.991086
759,1.163342,1.372769,0.975191,1.064057,1.179020,1.144023,1.150005,1.046793,1.162527
760,1.357032,1.628436,1.284077,1.052512,1.307512,1.238982,1.208221,0.945010,1.004646


In [34]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

# 假设 features 和 results 已经是 pandas.DataFrame
# features: (762, 25), results: (762, 9)

# Step 1: 构造标签（每行 cumulative return 最大的算法索引）
y = results.values.argmax(axis=1)   # shape (762,)
X = features.values                 # shape (762, 25)

# Step 2: 十折交叉验证
kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

train_returns, test_returns = [], []

for train_idx, test_idx in kf.split(X, y):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    # 标准化（Random Forest 不强制需要，但为了和其他模型对比保持一致）
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    
    # 训练 Random Forest 分类器
    clf = RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features='sqrt',
        bootstrap=True,
        random_state=42
    )
    clf.fit(X_train, y_train)
    
    # ---- 训练集累计收益 ----
    y_train_pred = clf.predict(X_train)
    train_cum_return = results.values[train_idx, y_train_pred].mean()
    train_returns.append(train_cum_return)
    
    # ---- 测试集累计收益 ----
    y_test_pred = clf.predict(X_test)
    test_cum_return = results.values[test_idx, y_test_pred].mean()
    test_returns.append(test_cum_return)
    
    # 保存最后一个模型用于特征重要性分析
    last_clf = clf

print("平均训练 cumulative return:", np.mean(train_returns))
print("平均测试 cumulative return:", np.mean(test_returns))

# 打印最重要的10个特征
print("\n最重要的10个特征:")
feature_importance = last_clf.feature_importances_
feature_names = features.columns
indices = np.argsort(feature_importance)[::-1][:10]
for i, idx in enumerate(indices):
    print(f"{i+1}. {feature_names[idx]}: {feature_importance[idx]:.4f}")



平均训练 cumulative return: 3.869328077528441
平均测试 cumulative return: 3.7701218605945144

最重要的10个特征:
1. max_var: 0.0334
2. mean_var: 0.0319
3. std_var: 0.0316
4. EVC standard deviation: 0.0316
5. Gross Margin standard deviation: 0.0312
6. ROA standard deviation: 0.0286
7. EVC mean: 0.0282
8. Period: 0.0273
9. std_avg: 0.0271
10. ROA mean: 0.0268


In [25]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier

# 假设 features 和 results 已经是 pandas.DataFrame
# features: (762, 25), results: (762, 9)

# Step 1: 构造标签（每行 cumulative return 最大的算法索引）
y = results.values.argmax(axis=1)   # shape (762,)
X = features.values                 # shape (762, 25)

# Step 2: 十折交叉验证
kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

train_returns, test_returns = [], []

for train_idx, test_idx in kf.split(X, y):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    # 标准化（MLP 对特征量纲敏感，必须标准化）
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    
    # 训练 MLP 分类器
    clf = MLPClassifier(
        hidden_layer_sizes=(100, 50),  # 两个隐藏层，分别有100和50个神经元
        activation='relu',             # 激活函数
        solver='adam',                 # 优化器
        alpha=0.0001,                  # L2正则化参数
        batch_size='auto',             # 批次大小
        learning_rate='constant',      # 学习率策略
        learning_rate_init=0.001,      # 初始学习率
        max_iter=500,                  # 最大迭代次数
        random_state=42
    )
    clf.fit(X_train, y_train)
    
    # ---- 训练集累计收益 ----
    y_train_pred = clf.predict(X_train)
    train_cum_return = results.values[train_idx, y_train_pred].mean()
    train_returns.append(train_cum_return)
    
    # ---- 测试集累计收益 ----
    y_test_pred = clf.predict(X_test)
    test_cum_return = results.values[test_idx, y_test_pred].mean()
    test_returns.append(test_cum_return)

print("平均训练 cumulative return:", np.mean(train_returns))
print("平均测试 cumulative return:", np.mean(test_returns))

c:\Users\zhy.LAPTOP-6LQS5T6S.000\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\zhy.LAPTOP-6LQS5T6S.000\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\zhy.LAPTOP-6LQS5T6S.000\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\zhy.LAPTOP-6LQS5T6S.000\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


平均训练 cumulative return: 3.92804368811161
平均测试 cumulative return: 3.6387144007348766


c:\Users\zhy.LAPTOP-6LQS5T6S.000\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


In [26]:
import numpy as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

# 假设 features 和 results 已经是 pandas.DataFrame
# features: (762, 25), results: (762, 9)

# Step 1: 构造标签（每行 cumulative return 最大的算法索引）
y = results.values.argmax(axis=1)   # shape (762,)
X = features.values                 # shape (762, 25)

# Step 2: 十折交叉验证
kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

train_returns, test_returns = [], []

for train_idx, test_idx in kf.split(X, y):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    # 标准化
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    
    # 训练 SVM
    clf = SVC(kernel='rbf', C=1, gamma='scale', random_state=42)
    clf.fit(X_train, y_train)
    
    # ---- 训练集累计收益 ----
    y_train_pred = clf.predict(X_train)
    train_cum_return = results.values[train_idx, y_train_pred].mean()
    train_returns.append(train_cum_return)
    
    # ---- 测试集累计收益 ----
    y_test_pred = clf.predict(X_test)
    test_cum_return = results.values[test_idx, y_test_pred].mean()
    test_returns.append(test_cum_return)

print("平均训练 cumulative return:", np.mean(train_returns))
print("平均测试 cumulative return:", np.mean(test_returns))


平均训练 cumulative return: 3.717000531465085
平均测试 cumulative return: 3.635038305696012


In [35]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

# 假设 features 和 results 已经是 pandas.DataFrame
# features: (762, 25), results: (762, 9)

# Step 1: 构造标签（每行 cumulative return 最大的算法索引）
y = results.values.argmax(axis=1)   # shape (762,)
X = features.values                 # shape (762, 25)

# Step 2: 十折交叉验证
kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

train_returns, test_returns = [], []

for train_idx, test_idx in kf.split(X, y):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    # 标准化（树模型不一定需要，但为了和 SVM 对比可以加上）
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    
    # 训练 XGBoost 分类器
    clf = XGBClassifier(
        objective='multi:softmax',  # 多分类
        num_class=results.shape[1], # 9
        eval_metric='mlogloss',
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        use_label_encoder=False
    )
    clf.fit(X_train, y_train)
    
    # ---- 训练集累计收益 ----
    y_train_pred = clf.predict(X_train)
    train_cum_return = results.values[train_idx, y_train_pred].mean()
    train_returns.append(train_cum_return)
    
    # ---- 测试集累计收益 ----
    y_test_pred = clf.predict(X_test)
    test_cum_return = results.values[test_idx, y_test_pred].mean()
    test_returns.append(test_cum_return)
    
    # 保存最后一个模型用于特征重要性分析
    last_clf = clf

print("平均训练 cumulative return:", np.mean(train_returns))
print("平均测试 cumulative return:", np.mean(test_returns))

# 打印最重要的10个特征
print("\n最重要的10个特征:")
feature_importance = last_clf.feature_importances_
feature_names = features.columns
indices = np.argsort(feature_importance)[::-1][:10]
for i, idx in enumerate(indices):
    print(f"{i+1}. {feature_names[idx]}: {feature_importance[idx]:.4f}")




c:\Users\zhy.LAPTOP-6LQS5T6S.000\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [15:46:36] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\zhy.LAPTOP-6LQS5T6S.000\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [15:46:38] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\zhy.LAPTOP-6LQS5T6S.000\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [15:46:38] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\zhy.LAPTOP-6LQS5T6S.000\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [15:46:39] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:7

平均训练 cumulative return: 3.9280703705427555
平均测试 cumulative return: 3.768065378214673

最重要的10个特征:
1. ROA mean: 0.0492
2. EVC standard deviation: 0.0371
3. RAPS standard deviation: 0.0350
4. SPS mean: 0.0322
5. P/B standard deviation: 0.0315
6. EV/EBITDA standard deviation: 0.0306
7. Net Margin mean: 0.0302
8. VEC mean: 0.0290
9. Gross Margin mean: 0.0281
10. IQS mean: 0.0279


In [28]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from lightgbm import LGBMClassifier

# 假设 features 和 results 已经是 pandas.DataFrame
# features: (762, 25), results: (762, 9)

# Step 1: 构造标签（每行 cumulative return 最大的算法索引）
y = results.values.argmax(axis=1)   # shape (762,)
X = features.values                 # shape (762, 25)

# Step 2: 十折交叉验证
kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

train_returns, test_returns = [], []

for train_idx, test_idx in kf.split(X, y):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    # 标准化（LightGBM 不强制需要，但为了和 SVM/XGBoost 对比保持一致）
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    
    # 训练 LightGBM 分类器
    clf = LGBMClassifier(
        objective='multiclass',
        num_class=results.shape[1], # 9 类
        n_estimators=200,
        max_depth=-1,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )
    clf.fit(X_train, y_train)
    
    # ---- 训练集累计收益 ----
    y_train_pred = clf.predict(X_train)
    train_cum_return = results.values[train_idx, y_train_pred].mean()
    train_returns.append(train_cum_return)
    
    # ---- 测试集累计收益 ----
    y_test_pred = clf.predict(X_test)
    test_cum_return = results.values[test_idx, y_test_pred].mean()
    test_returns.append(test_cum_return)

print("平均训练 cumulative return:", np.mean(train_returns))
print("平均测试 cumulative return:", np.mean(test_returns))


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000884 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6734
[LightGBM] [Info] Number of data points in the train set: 685, number of used features: 47
[LightGBM] [Info] Start training from score -3.162123
[LightGBM] [Info] Start training from score -1.975542
[LightGBM] [Info] Start training from score -1.492466
[LightGBM] [Info] Start training from score -4.044512
[LightGBM] [Info] Start training from score -1.996819
[LightGBM] [Info] Start training from score -3.095432
[LightGBM] [Info] Start training from score -2.418545
[LightGBM] [Info] Start training from score -2.865857
[LightGBM] [Info] Start training from score -1.387755
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[L

c:\Users\zhy.LAPTOP-6LQS5T6S.000\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\zhy.LAPTOP-6LQS5T6S.000\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000710 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6663
[LightGBM] [Info] Number of data points in the train set: 685, number of used features: 47
[LightGBM] [Info] Start training from score -3.128221
[LightGBM] [Info] Start training from score -1.986124
[LightGBM] [Info] Start training from score -1.492466
[LightGBM] [Info] Start training from score -4.044512
[LightGBM] [Info] Start training from score -1.996819
[LightGBM] [Info] Start training from score -3.095432
[LightGBM] [Info] Start training from score -2.418545
[LightGBM] [Info] Start training from score -2.865857
[LightGBM] [Info] Start training from score -1.387755
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[L

c:\Users\zhy.LAPTOP-6LQS5T6S.000\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\zhy.LAPTOP-6LQS5T6S.000\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

...

c:\Users\zhy.LAPTOP-6LQS5T6S.000\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\zhy.LAPTOP-6LQS5T6S.000\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000769 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6663
[LightGBM] [Info] Number of data points in the train set: 686, number of used features: 47
[LightGBM] [Info] Start training from score -3.163582
[LightGBM] [Info] Start training from score -1.977001
[LightGBM] [Info] Start training from score -1.487453
[LightGBM] [Info] Start training from score -4.045971
[LightGBM] [Info] Start training from score -1.998278
[LightGBM] [Info] Start training from score -3.096890
[LightGBM] [Info] Start training from score -2.420004
[LightGBM] [Info] Start training from score -2.867316
[LightGBM] [Info] Start training from score -1.389214
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[L

c:\Users\zhy.LAPTOP-6LQS5T6S.000\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\zhy.LAPTOP-6LQS5T6S.000\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001523 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6750
[LightGBM] [Info] Number of data points in the train set: 686, number of used features: 47
[LightGBM] [Info] Start training from score -3.163582
[LightGBM] [Info] Start training from score -1.977001
[LightGBM] [Info] Start training from score -1.487453
[LightGBM] [Info] Start training from score -4.045971
[LightGBM] [Info] Start training from score -1.998278
[LightGBM] [Info] Start training from score -3.096890
[LightGBM] [Info] Start training from score -2.420004
[LightGBM] [Info] Start training from score -2.867316
[LightGBM] [Info] Start training from score -1.389214
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[L

c:\Users\zhy.LAPTOP-6LQS5T6S.000\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\zhy.LAPTOP-6LQS5T6S.000\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [29]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from catboost import CatBoostClassifier

# 假设 features 和 results 已经是 pandas.DataFrame
# features: (762, 25), results: (762, 9)

# Step 1: 构造标签（每行 cumulative return 最大的算法索引）
y = results.values.argmax(axis=1)   # shape (762,)
X = features.values                 # shape (762, 25)

# Step 2: 十折交叉验证
kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

train_returns, test_returns = [], []

for train_idx, test_idx in kf.split(X, y):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    # 标准化（CatBoost 内部能处理不同量纲，但为了和 SVM/XGBoost/LightGBM 对齐可以保留）
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    
    # 训练 CatBoost 分类器
    clf = CatBoostClassifier(
        iterations=500,
        depth=6,
        learning_rate=0.05,
        loss_function='MultiClass',
        random_seed=42,
        verbose=0  # 不打印训练过程
    )
    clf.fit(X_train, y_train)
    
    # ---- 训练集累计收益 ----
    y_train_pred = clf.predict(X_train).astype(int).flatten()
    train_cum_return = results.values[train_idx, y_train_pred].mean()
    train_returns.append(train_cum_return)
    
    # ---- 测试集累计收益 ----
    y_test_pred = clf.predict(X_test).astype(int).flatten()
    test_cum_return = results.values[test_idx, y_test_pred].mean()
    test_returns.append(test_cum_return)

print("平均训练 cumulative return:", np.mean(train_returns))
print("平均测试 cumulative return:", np.mean(test_returns))


平均训练 cumulative return: 3.9044700057396
平均测试 cumulative return: 3.769778934118026


In [30]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

# 假设 features 和 results 已经是 pandas.DataFrame
# features: (762, 25), results: (762, 9)

# Step 1: 构造标签（每行 cumulative return 最大的算法索引）
y = results.values.argmax(axis=1)   # shape (762,)
X = features.values                 # shape (762, 25)

# Step 2: 十折交叉验证
kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

train_returns, test_returns = [], []

for train_idx, test_idx in kf.split(X, y):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    # kNN 对特征量纲敏感 → 必须标准化
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    
    # 训练 kNN 分类器
    clf = KNeighborsClassifier(
        n_neighbors=5,   # 常用默认值，可调
        weights='distance',  # 距离加权，离得近的邻居更重要
        metric='minkowski', # 默认欧式距离
        p=2
    )
    clf.fit(X_train, y_train)
    
    # ---- 训练集累计收益 ----
    y_train_pred = clf.predict(X_train)
    train_cum_return = results.values[train_idx, y_train_pred].mean()
    train_returns.append(train_cum_return)
    
    # ---- 测试集累计收益 ----
    y_test_pred = clf.predict(X_test)
    test_cum_return = results.values[test_idx, y_test_pred].mean()
    test_returns.append(test_cum_return)

print("平均训练 cumulative return:", np.mean(train_returns))
print("平均测试 cumulative return:", np.mean(test_returns))


平均训练 cumulative return: 3.9280703705427555
平均测试 cumulative return: 3.7143723868088627
